In [ ]:
import shap
import lime
import lime.lime_tabular
import numpy as np
import pandas as pd

class ModelExplainer:
    def init(self, model, feature_names, training_data=None):
        self.model = model
        self.feature_names = feature_names

        # 1. Initialize SHAP
        if self.model:
            self.shap_explainer = shap.TreeExplainer(self.model)
        else:
            self.shap_explainer = None

        # 2. Initialize LIME
        if training_data is not None:
            self.lime_explainer = lime.lime_tabular.LimeTabularExplainer(
                training_data=training_data,
                feature_names=self.feature_names,
                class_names=['Normal', 'Attack'],
                mode='classification'
            )
        else:
            self.lime_explainer = None

    def get_shap_explanation(self, scaled_features, top_n=3):
        if not self.shap_explainer:
            return {"Error": "SHAP Model not loaded"}
        try:
            data_for_shap = scaled_features.values if hasattr(scaled_features, 'values') else scaled_features
            shap_values = self.shap_explainer.shap_values(data_for_shap)

            # Extract impact values safely
            if isinstance(shap_values, list):
                attack_shap_values = shap_values[1][0]
            elif len(shap_values.shape) == 3:
                attack_shap_values = shap_values[0, :, 1]
            else:
                attack_shap_values = shap_values[0]

            feature_impacts = list(zip(self.feature_names, attack_shap_values))
            feature_impacts.sort(key=lambda x: abs(x[1]), reverse=True)

            explanation_dict = {}
            for feature_name, impact_value in feature_impacts[:top_n]:

                if impact_value > 0:
                    explanation_dict[feature_name] = f"Increased Attack Risk (+{impact_value:.2f})"
                else:
                    explanation_dict[feature_name] = f"Decreased Attack Risk ({impact_value:.2f})"
            return explanation_dict
        except Exception as e:
            return {"Alert": f"SHAP Error: {e}"}

    def get_lime_explanation(self, scaled_features_df, top_n=3):
        if not self.lime_explainer or not self.model:
            return {"Error": "LIME Explainer or Model not initialized"}

        try:
            predict_fn_wrapper = lambda x: self.model.predict_proba(pd.DataFrame(x, columns=self.feature_names))

            exp = self.lime_explainer.explain_instance(
                data_row=scaled_features_df.values[0],
                predict_fn=predict_fn_wrapper,
                num_features=top_n
            )

            explanation_dict = {}
            for feature_condition, impact_value in exp.as_list():

                if impact_value > 0:
                    explanation_dict[feature_condition] = f"Attack Indicator (+{impact_value:.2f})"
                else:
                    explanation_dict[feature_condition] = f"Normal Indicator ({impact_value:.2f})"

            return explanation_dict

        except Exception as e:
            return {"Alert": f"LIME Error: {e}"}
